In [1]:
import json
with open("./ReviewCritique_LLM.jsonl") as f:
    data = [json.loads(line) for line in f]

In [2]:
all_error_types = []
for data_item in data:
    for key in data_item:
        if key.startswith("review#") and data_item[key]:
            all_error_types.extend([i["error_type"] for i in data_item[key]["review"]])

In [3]:
import math
set([i for i in all_error_types if i is not None and type(i) != float])

set()

In [13]:
LLMs = ["claude_opus", "gpt4", "gemini_pro_1.5"]

In [35]:
import os
import hashlib

from cv2 import line
os.makedirs("./env/papers", exist_ok=True)
os.makedirs("./env/reviews", exist_ok=True)
os.makedirs("./env/answer_key", exist_ok=True)
reviews = {}
paper_ids = []
gt = {}
for i, item in enumerate(data):
    paper_id = hashlib.sha256(str(i).encode()).hexdigest()[:10]
    paper_ids.append(paper_id)
    with open(os.path.join("./env/papers", f"{paper_id}.md"), "w") as f:
        f.write(f"# {item['title']}\n\n")
        f.write(f"## Abstract\n\n{item['body_text']}\n\n")
    
    reviews[paper_id] = {}
    gt[paper_id] = {}
    for reviewer in [i for i in item if i.startswith("review#") or i in LLMs]:
        reviews[paper_id][reviewer] = "\n".join([f'{line}: {i["segment_text"]}' for line, i in enumerate(item[reviewer]["review"])])
    
    for reviewer in [i for i in item if i.startswith("review#") or i in LLMs]:
        gt[paper_id][reviewer] = "\n".join([f"{line}: {i['error_type']}" for line, i in enumerate(item[reviewer]["review"])])

In [69]:

prompt = """Your job is to judge if a paper review makes sense, judge each line individually (some line might not be review, just ignore those lines but still output line by line). Return a list of the judgement for each point of each reviewer and overall judgement of the reviewer. You also have access to the paper (and you should use it) itself for cross reference. Check {paper_path} for the paper, do not try to access other files. 
Common red flags to look for in the the whole review, mark each line with the corresponding error type if you find any of these issues:
| Error Type | Explanation |
|---|---|
| Misunderstanding | The reviewer misinterprets claims or ideas presented in the paper, leading to inaccurate or irrelevant comments. |
| Neglect | The reviewer overlooks important details explicitly stated in the paper, resulting in unwarranted questions or critiques. |
| Vague Critique | The review lacks specificity, claiming missing components without clearly identifying what is missing. |
| Out-of-scope | The reviewer suggests additional methods, experiments, or analyses that are beyond the intended scope of the paper. |
| Invalid Criticism | The reviewer's criticism is considered invalid, especially when suggesting impractical experiments or trivializing results. |
| Misinterpret Novelty | The reviewer questions the novelty of the work without substantiating their claims with relevant references. |
| Superficial Review | The reviewer appears to have only skimmed the paper, providing generic or unsupported comments about the presence or absence of weaknesses. |
| Writing | Discrepancies arise when the reviewer praises the writing, while you suggest it needs more clarity or explicitness. |
| Inexpert Statement | The reviewer exhibits a lack of domain knowledge, leading to unnecessary or irrelevant concerns. |
| Experiment | Conflicting opinions about the design of experiments; the reviewer praises them while you suggest adding more baselines or tests. |
| Unstated statement | Statements made in the review are not supported by content in the paper. |
| Contradiction | The reviewer contradicts themselves within the review, such as criticizing the paper's experiments while later stating that the experiments are comprehensive. |

Note that some points might have multiple issues, so mark all that apply. They could be normal (None) as well.

The review is as follows:
{review}
"""

In [70]:
from pydantic import BaseModel
class Line(BaseModel):
    line_number: int
    error_type: str
    confidence: float

class Result(BaseModel):
    line_judgement: list[Line]

In [71]:
from claude_agent_sdk import ClaudeAgentOptions, AssistantMessage, ResultMessage, TextBlock, ClaudeSDKClient
import random
chars = "⣾⣷⣯⣟⣻⣽⣾⣷⣯⣟⣻⣽"


async def judge(paper_id, reviewer=0):
    review = reviews[paper_id][list(reviews[paper_id].keys())[reviewer]]
    paper_path = f"./env/papers/{paper_id}.md"
    options = ClaudeAgentOptions(
    model="claude-opus-4-7",
    effort="high",
    allowed_tools=[f"Read({paper_path})"],
    output_format={
                "type": "json_schema",
                "schema": Result.model_json_schema(),
                },
    cwd="./")
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt.format(paper_path=paper_path, review=review))
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print(f"\nSession ID: {message.session_id}")
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print()
                        print(block.text)
            print(random.choice(chars), flush=True, end="")
    if isinstance(message, ResultMessage) and message.structured_output:
        result = Result.model_validate(message.structured_output)
        return result


In [76]:
idx = 6
print(gt[paper_ids[idx]]["claude_opus"])

0: nan
1: nan
2: nan
3: nan
4: nan
5: nan
6: nan
7: nan
8: nan
9: nan
10: nan
11: nan
12: nan
13: nan
14: nan
15: Out-of-scope
16: nan
17: nan
18: Out-of-scope
19: nan
20: nan
21: nan
22: Misinterpret Novelty
23: nan
24: nan
25: nan
26: nan
27: nan
28: nan
29: nan
30: nan


In [77]:
# print(reviews[paper_ids[idx]]["claude_opus"])

In [78]:
res = await judge(paper_ids[idx]) 

⣟⣽⣷⣾⣷⣾⣟⣻⣻⣯⣽⣾⣷⣯⣷⣯⣷⣻
I've analyzed the review line-by-line against the paper. Key findings:

- **Lines 1-14**: Summary and strengths sections accurately reflect the paper's claims (1/7 parameters, 1/48 tiny version, 96.7% performance, 2.7x speedup, Tucker decomposition comparison, complementarity to KD).
- **Line 15**: Critiquing GLUE-only evaluation is **Out-of-scope**, as the paper explicitly focuses on BERT compression using the standard GLUE benchmark.
- **Line 16**: "Theoretical analysis limited" is a **Vague Critique** — the reviewer doesn't specify what theory is missing; the paper actually provides motivation via decomposability analysis (Sec. 3) and PCA experiments.
- **Line 17**: Claiming the paper doesn't "explore the limits of compression" is **Invalid Criticism** — the paper's explicit theme is *extreme* compression and already demonstrates 1/48 parameter compression.
- **Line 18**: Requesting pre-training efficiency analysis is **Out-of-scope** — the paper compresses alread

In [79]:
lines = json.loads(res.model_dump_json())["line_judgement"]
positive_lines = [f'{line["line_number"]}, {line["error_type"]}' for line in lines if line["error_type"] != "None" and line["confidence"] > 0.5]
positive_lines 

['15, Out-of-scope',
 '16, Vague Critique',
 '17, Invalid Criticism',
 '18, Out-of-scope',
 '28, Out-of-scope']